# 02: Chronological Data Preparation

Prepare a selected hardware group for modeling: load its available member systems, resolve Notebook 1's shared feature registry, protect targets from imputation, split chronologically, fit preprocessing on training data only, and export group-named CSV matrices.

The selected registry may include `poa_irradiance`; it is treated as a target here and excluded from `X`. Member datasets are aligned by timestamp and selected feature name before concatenation.

## Hardware Groups and IDs

Choose `GROUP_ID` in the execution cell. The pipeline combines available member-system files using the shared feature registry. An explicit system override is optional and must be a member of the selected group.

| GROUP_ID | Group name | Systems |
|---:|---|---|
| 1 | `group_base_4_1283_1289` | system 4, system 1283, system 1289 |
| 2 | `group_34_35` | system 34, system 35 |
| 3 | `group_50_51` | system 50, system 51 |
| 4 | `group_1276_1277` | system 1276, system 1277 |
| 5 | `group_10` | system 10 |
| 6 | `group_33` | system 33 |
| 7 | `group_36` | system 36 |
| 8 | `group_1200` | system 1200 |

For groups with multiple systems, selected features absent from one member are imputed from training-partition means after all available member datasets are combined.

In [ ]:
from pathlib import Path
import importlib
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "data" / "preparation.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the solarMLthesis project root")
sys.path.insert(0, str(PROJECT_ROOT))

import src.data.preparation
importlib.reload(src.data.preparation)
from src.data.preparation import prepare_group_features

GROUP_ID = 8
outputs = prepare_group_features(group_id=GROUP_ID)

In [15]:
import pandas as pd
import pickle
from pathlib import Path

def audit_and_export_splits(project_root: str = ".", output_csv: str = "experiment_summary.csv") -> pd.DataFrame:
    feature_dir = Path(project_root) / "data" / "features"
    model_dir = Path(project_root) / "artifacts" / "models"
    output_path = Path(project_root) / output_csv
    
    summary_records = []
    
    if not feature_dir.exists():
        raise FileNotFoundError(f"Feature directory not found at: {feature_dir}")
        
    for group_path in sorted(feature_dir.glob("*")):
        if not group_path.is_dir():
            continue
            
        group_id = group_path.name
        record = {
            "group_id": group_id,
            "X_train_shape": None, "X_test_shape": None,
            "y_train_shape": None, "y_test_shape": None,
            "train_time_min": None, "train_time_max": None,
            "test_time_min": None, "test_time_max": None,
            "total_nans": 0, "scaler_type": None, "scaler_n_features": None
        }
        
        # Audit suffixed CSV matrices
        for split in ["X_train", "X_test", "y_train", "y_test"]:
            file_path = group_path / f"{split}_{group_id}.csv"
            if file_path.exists():
                # Assumes the first column is the datetime index
                df = pd.read_csv(file_path, index_col=0, parse_dates=True)
                record[f"{split}_shape"] = str(df.shape)
                record["total_nans"] += int(df.isna().sum().sum())
                
                if split == "X_train":
                    record["train_time_min"] = df.index.min()
                    record["train_time_max"] = df.index.max()
                elif split == "X_test":
                    record["test_time_min"] = df.index.min()
                    record["test_time_max"] = df.index.max()
        
        # Audit Serialized Scaler
        scaler_path = model_dir / f"scaler_{group_id}.pkl"
        if scaler_path.exists():
            with open(scaler_path, "rb") as f:
                scaler = pickle.load(f)
            record["scaler_type"] = type(scaler).__name__
            record["scaler_n_features"] = getattr(scaler, "n_features_in_", None)
        
        summary_records.append(record)
    
    summary_df = pd.DataFrame(summary_records)
    summary_df.to_csv(output_path, index=False)
    return summary_df

if __name__ == "__main__":
    # Point project_root to the parent directory (repository root)
    df_summary = audit_and_export_splits(project_root="..")
    print(df_summary.head())

          group_id X_train_shape X_test_shape y_train_shape y_test_shape  \
0         group_10   (24258, 12)   (6065, 12)    (24258, 2)    (6065, 2)   
1       group_1200   (15420, 25)   (3855, 25)    (15420, 2)    (3855, 2)   
2  group_1276_1277    (26868, 6)    (6717, 6)    (26868, 2)    (6717, 2)   
3         group_33   (43787, 13)  (10947, 13)    (43787, 2)   (10947, 2)   
4      group_34_35    (27534, 6)    (6884, 6)    (27534, 2)    (6884, 2)   

       train_time_min      train_time_max       test_time_min  \
0 2008-01-01 00:00:00 2008-10-29 19:30:00 2008-10-29 19:45:00   
1 2011-01-01 07:30:06 2011-10-16 15:15:48 2011-10-16 15:30:48   
2 2015-01-01 00:00:00 2015-10-09 05:45:00 2015-10-09 06:00:00   
3 2022-01-01 00:00:00 2022-01-31 09:46:00 2022-01-31 09:47:00   
4 2015-01-01 00:00:00 2015-10-18 05:30:00 2015-10-18 05:45:00   

        test_time_max  total_nans   scaler_type  scaler_n_features  
0 2008-12-31 23:45:00           0  MinMaxScaler                 12  
1 2011-12-31 1